## 40 分钟课堂案例：同一批邮件，三种分类模型

**对应课件主题**：逻辑回归（概率与阈值）、K 近邻（距离、尺度与 K）、朴素贝叶斯（计数、先验、平滑）。

教学用合成数据共 600 封英文邮件，标签 `spam`/`ham`；它不代表真实邮件分布，也不应用于实际过滤。按标题搜索 `C01`～`C29`，与讲稿一一对应。现场可逐段运行；时间紧时课前先 Run All，再现场重跑标记的代码单元。

**Kaggle 用法**：上传本 `.ipynb`，点击 Run All；若希望单独使用 CSV，把 `teaching_emails.csv` 加入 Input。未挂载 CSV 时，C03 使用固定种子生成逐行相同的数据。无需联网下载和安装额外依赖。

## 0–4 分钟｜导入与数据

目标：说清输入是邮件文本、输出是两类标签，辨认数据泄漏与训练/测试分离。

In [ ]:
# C01｜环境与随机种子
import csv, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             precision_score, recall_score, f1_score, precision_recall_curve,
                             roc_auc_score)
SEED = 2026
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (7, 4)
print('已准备：numpy / pandas / matplotlib / scikit-learn')


In [ ]:
# C02｜生成器：独立 CSV 缺席时也能运行
def generate_emails(seed=2026, n=600):
    rng = random.Random(seed)
    labels = ["spam"] * 210 + ["ham"] * (n - 210)
    rng.shuffle(labels)
    subject_spam = ["Limited offer today", "Account notice", "Your invoice update", "Project follow up", "A quick reminder", "Exclusive savings", "Action needed today", "Monthly update"]
    subject_ham = ["Project meeting", "Account notice", "Your invoice update", "Weekend plans", "A quick reminder", "Team report", "Monthly update", "Schedule for Friday"]
    spam_core = ["claim your bonus with this private link", "limited discount for selected members", "verify your account to keep access", "special price ends this evening", "get a free trial with the new plan", "urgent reply requested for your reward", "confirm payment details to finish the order", "exclusive invitation with a limited offer"]
    ham_core = ["please review the project notes before our meeting", "the team shared the latest report", "can we move the call to Friday afternoon", "thank you for sending the invoice", "your order receipt is attached for records", "the workshop agenda is ready to review", "let us confirm the schedule tomorrow", "please check the draft and send feedback"]
    shared = ["let me know if you have questions", "please reply when convenient", "we will send an update soon", "the details are included below", "thank you for your time", "contact the team if you need help"]
    confusing = ["the newsletter has a free webinar and a discount code", "our team needs an urgent account update", "your colleague sent a link to the shared report", "please verify the meeting time in the calendar"]
    rows = []
    for i, label in enumerate(labels, 1):
        core = spam_core if label == "spam" else ham_core
        alt = ham_core if label == "spam" else spam_core
        subject = rng.choice(subject_spam if label == "spam" else subject_ham)
        body_parts = [rng.choice(core), rng.choice(shared)]
        if rng.random() < .22: body_parts.append(rng.choice(confusing))
        if rng.random() < .13: body_parts.append(rng.choice(alt))
        rng.shuffle(body_parts)
        rows.append({"email_id": f"E{i:04d}", "subject": subject,
                     "body": ". ".join(body_parts) + ".", "label": label})
    return rows


In [ ]:
# C03｜优先读取 Kaggle Input，否则生成相同数据
paths = sorted(Path('/kaggle/input').rglob('teaching_emails.csv')) if Path('/kaggle/input').exists() else []
if not paths and Path('teaching_emails.csv').exists(): paths = [Path('teaching_emails.csv')]
df = pd.read_csv(paths[0]) if paths else pd.DataFrame(generate_emails(SEED))
assert list(df.columns) == ['email_id', 'subject', 'body', 'label']
assert len(df) == 600 and set(df.label) == {'ham', 'spam'} and not df.isna().any().any()
df['text'] = df['subject'] + ' ' + df['body']
print('数据来源：', str(paths[0]) if paths else '固定种子内置生成器')
print(df[['email_id','subject','body','label']].head(4).to_string(index=False))


In [ ]:
# C04｜看类比例与文本长度
print(df.label.value_counts().to_string())
print('垃圾邮件占比：', round(df.label.eq('spam').mean(), 3))
df.assign(words=df.text.str.split().str.len()).groupby('label')['words'].agg(['mean','min','max']).round(1).plot(kind='bar', rot=0)
plt.title('Word counts by class (synthetic data)'); plt.ylabel('Words'); plt.tight_layout(); plt.show()


In [ ]:
# C05｜分层划分；此后封存测试集
X_train, X_test, y_train, y_test = train_test_split(
    df.text, df.label.eq('spam').astype(int), test_size=.20,
    stratify=df.label, random_state=SEED)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
print('训练：', len(X_train), y_train.value_counts().to_dict())
print('测试：', len(X_test), y_test.value_counts().to_dict())
print('向量器只在训练折上 fit；测试集在最终选择前不查看。')


## 4–12 分钟｜逻辑回归：分数、概率、阈值

公式：$p=1/(1+e^{-z})$，其中 $z=w_0+\sum_i w_i x_i$。词被 TF-IDF 转成数值后，模型给出类 `spam` 的概率。阈值决定行动，概率本身不是标签。

In [ ]:
# C06｜画 Sigmoid 曲线
z = np.linspace(-6, 6, 301)
p = 1/(1+np.exp(-z))
plt.plot(z,p); plt.axhline(.5,color='gray',ls='--'); plt.axvline(0,color='gray',ls='--')
plt.xlabel('Linear score z'); plt.ylabel('P(spam)'); plt.title('Sigmoid: score to probability')
plt.tight_layout(); plt.show()
print('z=-2,0,2 的概率：', np.round(1/(1+np.exp(-np.array([-2,0,2]))),3))


In [ ]:
# C07｜阈值的手算小实验
toy_probs = np.array([.12, .36, .51, .74, .91])
print(pd.DataFrame({'P(spam)':toy_probs,
    'threshold_0.30':(toy_probs>=.30).astype(int),
    'threshold_0.50':(toy_probs>=.50).astype(int),
    'threshold_0.80':(toy_probs>=.80).astype(int)}).to_string(index=False))


In [ ]:
# C08｜训练文本逻辑回归
lr = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2)),
               ('model', LogisticRegression(max_iter=1000, random_state=SEED))])
lr_cv = cross_val_score(lr, X_train, y_train, cv=cv, scoring='f1')
print('4 折 F1：', lr_cv.round(3), '均值：', round(lr_cv.mean(),3))
lr.fit(X_train, y_train)


In [ ]:
# C09｜查几个词对结果的方向
vocab = lr.named_steps['tfidf'].get_feature_names_out()
weights = lr.named_steps['model'].coef_[0]
order = np.argsort(weights)
print('偏向 ham：', list(zip(vocab[order[:7]],np.round(weights[order[:7]],2))))
print('偏向 spam：', list(zip(vocab[order[-7:]],np.round(weights[order[-7:]],2))))


In [ ]:
# C10｜用训练集的交叉验证预测比较阈值
oof_prob = cross_val_predict(lr, X_train, y_train, cv=cv, method='predict_proba')[:,1]
threshold_rows = []
for t in [.25, .40, .50, .65, .80]:
    pred = (oof_prob >= t).astype(int)
    threshold_rows.append({'threshold':t,
        'precision':precision_score(y_train,pred,zero_division=0),
        'recall':recall_score(y_train,pred,zero_division=0),
        'f1':f1_score(y_train,pred,zero_division=0)})
threshold_table = pd.DataFrame(threshold_rows)
print(threshold_table.round(3).to_string(index=False))


In [ ]:
# C11｜固定阈值的选择规则
# 课堂策略：在候选阈值中，先保证训练折预测的 precision >= .80，再选 recall 最大者；同分选较高阈值。
eligible = threshold_table[threshold_table.precision >= .80]
if len(eligible):
    chosen_threshold = float(eligible.sort_values(['recall','threshold'],ascending=[False,False]).iloc[0].threshold)
else:
    chosen_threshold = float(threshold_table.sort_values(['f1','threshold'],ascending=[False,False]).iloc[0].threshold)
print('训练阶段固定阈值：', chosen_threshold)


## 21 分钟前要停一停

提问：如果把阈值调低，通常会发生什么？预期：预测为垃圾的邮件增多，召回通常上升，误拦也可能增多。不要把训练折分数当作独立测试表现。

## 12–21 分钟｜K 近邻：距离、尺度与 K

KNN 不学习显式公式：新邮件向量靠近哪些训练邮件，就参考其标签投票。文本向量高维稀疏；先用二维小例子看懂，再回到同一数据集。

In [ ]:
# C12｜二维邻居与距离
points = np.array([[1,1],[2,1],[2,3],[4,3],[5,4],[5,5]],float)
point_labels = np.array([0,0,0,1,1,1])
new_point = np.array([3.6,3.2])
distances = np.sqrt(((points-new_point)**2).sum(axis=1))
order = np.argsort(distances)
print(pd.DataFrame({'x':points[:,0],'y':points[:,1],'label':point_labels,'distance':distances})
      .iloc[order].round(2).to_string(index=False))
plt.scatter(points[:,0],points[:,1],c=point_labels,cmap='coolwarm',s=90,edgecolors='k')
plt.scatter(*new_point,marker='*',s=270,c='gold',edgecolors='k'); plt.title('Nearest neighbours (illustrative 2D)')
plt.tight_layout(); plt.show()


In [ ]:
# C13｜现场手数 K=1、3、5 的票
for k in [1,3,5]:
    votes = point_labels[order[:k]]
    print(f'K={k}: neighbours={votes.tolist()}, spam votes={votes.sum()}, prediction={int(votes.mean() >= .5)}')


In [ ]:
# C14｜改变尺度，最近邻可能改变
scale_points = np.array([[0,4],[3,0],[8,20]],float)
scale_query = np.array([[1,0]],float)
raw = np.linalg.norm(scale_points-scale_query,axis=1)
scaler_demo = StandardScaler().fit(scale_points)
scaled = np.linalg.norm(scaler_demo.transform(scale_points)-scaler_demo.transform(scale_query),axis=1)
print('原始距离：',raw.round(2),'最近邻：',int(raw.argmin()))
print('标准化后：',scaled.round(2),'最近邻：',int(scaled.argmin()))
assert raw.argmin() != scaled.argmin()
print('这只是尺度现象演示；实际建模时 scaler 也必须放在训练折的 pipeline 内。')


In [ ]:
# C15｜同一邮件数据上，交叉验证选择 K
knn_rows = []
for k in [1,3,5,9,15]:
    pipe = Pipeline([('tfidf', TfidfVectorizer(min_df=2)),
                     ('model', KNeighborsClassifier(n_neighbors=k, metric='cosine'))])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    knn_rows.append({'K':k,'f1_mean':scores.mean(),'f1_std':scores.std()})
knn_table = pd.DataFrame(knn_rows)
print(knn_table.round(3).to_string(index=False))
best_k = int(knn_table.sort_values(['f1_mean','K'],ascending=[False,True]).iloc[0].K)


In [ ]:
# C16｜拟合选中的 KNN
knn = Pipeline([('tfidf',TfidfVectorizer(min_df=2)),
                ('model',KNeighborsClassifier(n_neighbors=best_k, metric='cosine'))])
knn.fit(X_train,y_train)
print('选定 K =',best_k)


## 21–32 分钟｜朴素贝叶斯：从手算到邮件

模型将词出现次数作为证据；朴素的假设是给定类别时各词条件独立。这里用一个独立的小型体育文本算例展示拉普拉斯平滑，再把同样思想用于邮件。小算例不宣称与原课件句子完全一致。

In [ ]:
# C17｜五句体育小语料：类别、计数
sport_docs = ['team wins match', 'team scores goal', 'coach plans match',
              'stock market rises', 'market prices fall']
sport_y = np.array([1,1,1,0,0]) # 1=sport, 0=other
toy_vec = CountVectorizer()
toy_counts = toy_vec.fit_transform(sport_docs)
print('词表：', toy_vec.get_feature_names_out().tolist())
print(pd.DataFrame(toy_counts.toarray(), columns=toy_vec.get_feature_names_out()).assign(sport=sport_y).to_string(index=False))


In [ ]:
# C18｜先验和加一平滑手算
query = 'team match goal'
q_counts = toy_vec.transform([query]).toarray()[0]
V = len(toy_vec.get_feature_names_out())
log_scores = {}
for cls in [0,1]:
    class_counts = toy_counts.toarray()[sport_y==cls].sum(axis=0)
    prior = (sport_y==cls).mean()
    likelihoods = (class_counts+1)/(class_counts.sum()+V)
    log_scores[cls] = np.log(prior) + np.dot(q_counts,np.log(likelihoods))
    print(f'class={cls}: prior={prior:.3f}, total_tokens={class_counts.sum()}, denominator={class_counts.sum()+V}, log_score={log_scores[cls]:.4f}')
manual_sport_p = np.exp(log_scores[1])/(np.exp(log_scores[0])+np.exp(log_scores[1]))
print('P(sport | query), 手算 =',round(manual_sport_p,6))


In [ ]:
# C19｜与 sklearn 的 MultinomialNB 逐项核对
toy_nb = MultinomialNB(alpha=1).fit(toy_counts,sport_y)
sklearn_sport_p = toy_nb.predict_proba(toy_vec.transform([query]))[0,1]
print('scikit-learn =',round(sklearn_sport_p,6),' 差值 =',abs(sklearn_sport_p-manual_sport_p))
assert np.isclose(manual_sport_p,sklearn_sport_p)


In [ ]:
# C20｜对比 alpha 的效果
unseen_query = 'team market'
for alpha in [.1,1,10]:
    model = MultinomialNB(alpha=alpha).fit(toy_counts,sport_y)
    print(f'alpha={alpha:>4}: P(sport | {unseen_query!r})={model.predict_proba(toy_vec.transform([unseen_query]))[0,1]:.3f}')


In [ ]:
# C21｜先验变了，后验也会变
for sport_prior in [.2,.5,.8]:
    model = MultinomialNB(alpha=1,fit_prior=False,class_prior=[1-sport_prior,sport_prior]).fit(toy_counts,sport_y)
    p = model.predict_proba(toy_vec.transform(['match']))[0,1]
    print(f'先验 P(sport)={sport_prior:.1f}，观察 match 后 P(sport)={p:.3f}')


In [ ]:
# C22｜邮件任务：在训练集选平滑强度
nb_rows=[]
for alpha in [.1,1,10]:
    pipe = Pipeline([('count',CountVectorizer(ngram_range=(1,2),min_df=2)),
                     ('model',MultinomialNB(alpha=alpha))])
    scores = cross_val_score(pipe,X_train,y_train,cv=cv,scoring='f1')
    nb_rows.append({'alpha':alpha,'f1_mean':scores.mean(),'f1_std':scores.std()})
nb_table=pd.DataFrame(nb_rows)
print(nb_table.round(3).to_string(index=False))
best_alpha=float(nb_table.sort_values(['f1_mean','alpha'],ascending=[False,True]).iloc[0].alpha)


In [ ]:
# C23｜拟合选中的朴素贝叶斯
nb=Pipeline([('count',CountVectorizer(ngram_range=(1,2),min_df=2)),
             ('model',MultinomialNB(alpha=best_alpha))])
nb.fit(X_train,y_train)
print('选定 alpha =',best_alpha)


## 32–38 分钟｜独立测试与错误复盘

三个模型的参数与逻辑回归阈值都已根据训练数据定好，现在只评估一次测试集。注意：先前的交叉验证成绩和此处的独立测试成绩含义不同。

In [ ]:
# C24｜一次性评估三个模型
models={'Logistic Regression':lr,'KNN':knn,'Multinomial NB':nb}
test_predictions={}
rows=[]
for name,model in models.items():
    probs=model.predict_proba(X_test)[:,1]
    threshold=chosen_threshold if name=='Logistic Regression' else .5
    pred=(probs>=threshold).astype(int)
    test_predictions[name]=pred
    rows.append({'model':name,'threshold':threshold,
                 'precision':precision_score(y_test,pred,zero_division=0),
                 'recall':recall_score(y_test,pred,zero_division=0),
                 'f1':f1_score(y_test,pred,zero_division=0),
                 'roc_auc':roc_auc_score(y_test,probs)})
results=pd.DataFrame(rows)
print(results.round(3).to_string(index=False))


In [ ]:
# C25｜混淆矩阵：说清误拦与漏拦
fig,axes=plt.subplots(1,3,figsize=(12,3.6))
for ax,(name,pred) in zip(axes,test_predictions.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test,pred),display_labels=['ham','spam']).plot(ax=ax,colorbar=False,values_format='d')
    ax.set_title(name)
plt.tight_layout();plt.show()


In [ ]:
# C26｜逐类报告与常见错误
print(classification_report(y_test,test_predictions['Logistic Regression'],target_names=['ham','spam'],zero_division=0))
err=pd.DataFrame({'text':X_test,'actual':y_test,'pred':test_predictions['Logistic Regression']},index=X_test.index)
err=err[err.actual!=err.pred].copy()
err['type']=np.where((err.actual==0)&(err.pred==1),'误拦正常邮件','漏拦垃圾邮件')
print('错例数：',len(err))
print(err[['type','text']].head(6).to_string(index=False,max_colwidth=100))


In [ ]:
# C27｜真正现场输入两封新邮件
examples=['Team meeting on Friday please review the draft report',
          'Limited offer claim your free trial and confirm payment today']
live_probs=lr.predict_proba(examples)[:,1]
print(pd.DataFrame({'email':examples,'P(spam)':live_probs.round(3),
                    'decision':np.where(live_probs>=chosen_threshold,'spam','ham')}).to_string(index=False))


## 38–40 分钟｜出口题

快速作答：① 为什么向量器不能在完整数据上 `fit`？② 把阈值从 0.5 降到 0.25，通常哪类错误可能增加？③ KNN 为什么关心尺度？④ 朴素贝叶斯的 `alpha` 对未见词有什么作用？

In [ ]:
# C28｜出口题：阈值再试一次（仅训练折预测）
for t in [.25,.50]:
    cm=confusion_matrix(y_train,(oof_prob>=t).astype(int))
    print(f't={t}: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')


In [ ]:
# C29｜复核数据与结果，不重新调参
assert len(df)==600 and len(X_train)==480 and len(X_test)==120
assert X_train.index.intersection(X_test.index).empty
assert np.isclose(manual_sport_p,sklearn_sport_p)
assert set(results.model)=={'Logistic Regression','KNN','Multinomial NB'}
print('全部检查通过；课堂可修改 C07、C13、C20、C28 中的值再预测。')


## 讲者备注与练习答案

1. 若在划分前对全部邮件拟合词表或标准化器，测试集的信息会进入训练步骤。Pipeline 可让每个交叉验证折独立拟合。
2. 阈值变低通常增加阳性预测，漏拦可能减少、误拦可能增加；具体数字看 C28。
3. 距离依赖各特征的单位和幅度；尺度大的列可能左右结果。
4. 拉普拉斯平滑给未见词非零条件概率；`alpha` 越大，概率越往均匀分布收缩。

**延伸作业**：改变阈值候选和误拦代价；报告两类错误的数量。真实部署还须验证数据代表性、时间漂移、概率校准和成本。